# 전체 가맹점 원본 vs 근거 기반 보강 비교

기존 노트북을 수정하지 말고 이 노트북을 새로 열어 **1~6 순서대로** 실행하세요. 회사 데이터의 Colab 업로드 허용 여부는 기존 보안 정책을 따르세요.

- 전체 9,354개 가맹점을 두 버전 모두 검색 대상으로 사용합니다. 장소 필터·시장명 질의는 제외합니다.
- 원본 검색문서와 보강문서는 제공 ZIP 안의 CSV에 함께 있습니다.
- 보강 라벨은 규칙 기반 후보이지 정답이 아닙니다. 알려진 검색결과 판정은 별도 qrels로 관리합니다.
- 20개 질의는 이미 보고 개선 방향을 정한 **개발용 질의**입니다. 최종 모델 선정에는 새로운 공통 질의가 필요합니다.
- 기존 CSV·라벨·노트북을 덮어쓰지 않습니다. 이 노트북 자체의 출력을 Git에 올리지 마세요.


## 1. GPU 설정 및 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**. 아래 셀을 실행합니다.

In [ ]:
%pip -q install FlagEmbedding


## 2. 입력 ZIP 업로드
함께 제공한 **full_corpus_comparison_input.zip 하나만** 업로드하세요. 원본 Excel이나 기존 baseline CSV를 추가 업로드할 필요 없습니다.

In [ ]:
import io, json, hashlib, zipfile, time, platform, importlib.metadata
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('입력 ZIP 하나만 선택하세요.')
payload = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    expected = ['search_documents_enriched_v1.csv', 'evaluation_queries.csv', 'prior_qrels.csv']
    if not all(n in z.namelist() for n in expected):
        raise ValueError('제공된 full_corpus_comparison_input.zip을 선택하세요.')
    data, queries, qrels = [pd.read_csv(io.BytesIO(z.read(n)), dtype=str, keep_default_na=False) for n in expected]
assert len(data) == 9354 and data.record_id.is_unique
assert queries.query_id.is_unique and len(queries) == 20
assert not queries.query_type.eq('market_item').any()
assert not qrels.duplicated(['query_id', 'record_id']).any()
print('검색 대상:', len(data), '질의:', len(queries), '기존 판정:', qrels.relevance.ne('').sum())
display(data[['merchant_name','original_items','search_document','search_document_enriched']].head())


## 3. 저장 위치와 모델 로딩
기본 저장소는 Colab 임시 디스크입니다. 런타임 삭제 시 사라집니다. 회사 정책상 Drive 보관이 허용되고 지속 저장을 원할 때만 `USE_DRIVE=True`로 바꾸세요. 아래 캐시는 내용·모델·설정으로 구분되므로 다른 버전의 벡터를 잘못 재사용하지 않습니다.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive/merchant_search_comparison')
else:
    root = Path('/content/merchant_search_comparison')
root.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임으로 변경한 뒤 1번부터 실행하세요.')
from FlagEmbedding import BGEM3FlagModel
MODEL_NAME = 'BAAI/bge-m3'
MAX_LENGTH = 128
BATCH_SIZE = 32
TOP_K = 5
model = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
settings = dict(model=MODEL_NAME, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
                use_fp16=True, top_k=TOP_K, retrieval='exact_cosine_no_filter',
                model_revision=getattr(model.model.config, '_commit_hash', None),
                torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
                FlagEmbedding=importlib.metadata.version('FlagEmbedding'),
                transformers=importlib.metadata.version('transformers'),
                python=platform.python_version())
print(settings)


## 4. 전체 데이터 두 버전 임베딩·검색
원본과 보강본을 **같은 모델·질의·설정**으로 실행합니다. 문서 512건 단위로 캐시를 저장합니다. Qdrant 대신 정규화 벡터의 전체 코사인 검색을 사용해 DB 설정 차이를 제거합니다. 상호명/시장 메타데이터는 결과 확인용이며 시장 필터는 없습니다.

In [ ]:
def encode_cached(texts):
    key = hashlib.sha256(json.dumps([settings, texts], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    folder = root / 'cache' / key
    folder.mkdir(parents=True, exist_ok=True)
    blocks = []
    for start in range(0, len(texts), 512):
        file = folder / f'{start}.npy'
        if file.exists():
            block = np.load(file, allow_pickle=False)
        else:
            block = model.encode(texts[start:start+512], batch_size=BATCH_SIZE,
                max_length=MAX_LENGTH, return_dense=True, return_sparse=False,
                return_colbert_vecs=False)['dense_vecs'].astype(np.float32)
            norm = np.linalg.norm(block, axis=1, keepdims=True)
            if not np.isfinite(block).all() or (norm <= 0).any():
                raise ValueError('Invalid embeddings')
            block = block / norm
            temp = folder / f'{start}.tmp.npy'
            np.save(temp, block)
            temp.replace(file)
        assert block.shape == (min(512, len(texts)-start), 1024)
        blocks.append(block)
    return np.concatenate(blocks)

qvec = encode_cached(queries['query'].tolist())
vectors = {}
frames = []
timings = {}
for version, column in [('raw','search_document'),('enriched_v1','search_document_enriched')]:
    start = time.perf_counter()
    vectors[version] = encode_cached(data[column].tolist())
    timings[version] = time.perf_counter()-start
    scores = qvec @ vectors[version].T
    for i, q in queries.iterrows():
        positions = np.argsort(-scores[i], kind='stable')[:TOP_K]
        result = data.iloc[positions][['record_id','merchant_name','original_items','market_name']].copy()
        result.insert(0,'score',scores[i,positions])
        result.insert(0,'rank',np.arange(1,TOP_K+1))
        for name in ['query_type','query','query_id']:
            result.insert(0,name,q[name])
        result.insert(0,'data_version',version)
        result.insert(0,'model',MODEL_NAME)
        result.insert(0,'experiment_id','full_corpus_comparison_v1')
        frames.append(result)
    partial = pd.concat(frames, ignore_index=True)
    partial.to_csv(root/'comparison_results.csv', index=False, encoding='utf-8-sig')
results = pd.concat(frames, ignore_index=True)
settings['embedding_seconds_including_cache'] = timings
settings['input_zip_sha256'] = hashlib.sha256(payload).hexdigest()
(root/'run_manifest.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2))
display(results[results.query_id.eq('Q006')])


## 5. 기존 판정 연결·잠정 지표·결과 다운로드
보강에 사용한 규칙으로 정답을 생성하지 않습니다. 과거 판정과 질의ID+가맹점ID가 같은 경우만 연결합니다. 새 결과는 미판정으로 남습니다.

`judged_at_5`는 상위 5개 중 판정 보유 비율입니다. `unknown_as_nonmatch`와 `unknown_as_match`는 미판정을 각각 불일치/일치로 가정한 민감도 진단입니다. **신뢰구간이나 확정 성능이 아니며 AI 판정 자체의 오류는 포함하지 않습니다.** `relevance=2`만 정확한 일치로 계산합니다.

원본·보강본의 새 후보를 합친 `pooled_results_for_review.csv`를 함께 저장합니다. 다운로드 ZIP을 대화에 전달하면 새 후보도 검토할 수 있습니다. 미판정을 0으로 채우지 마세요.

In [ ]:
"""Scoring fixed query-document qrels; no use of enrichment rules or scores as labels."""
from collections import defaultdict
from statistics import mean


def score_results(rows, qrels, k=5):
    if k != 5 or not rows:
        raise ValueError('This comparison requires nonempty results and k=5')
    lookup = {}
    for r in qrels:
        key = (str(r['query_id']), str(r['record_id']))
        if key in lookup:
            raise ValueError('Duplicate query-document judgment')
        raw = str(r.get('relevance', '')).strip()
        grade = None if not raw else float(raw)
        if grade is not None and grade not in (0, 1, 2):
            raise ValueError('Invalid relevance')
        if grade is not None and not r.get('label_source'):
            raise ValueError('A label requires its provenance')
        lookup[key] = grade
    groups = defaultdict(list)
    for row in rows:
        groups[(row['data_version'], row['query_id'])].append(row)
    result = []
    for (version, qid), group in sorted(groups.items()):
        group = sorted(group, key=lambda r: int(r['rank']))
        if len({r['record_id'] for r in group}) != len(group):
            raise ValueError('Duplicate result ID')
        top = group[:k]
        if [int(r['rank']) for r in top] != list(range(1, k + 1)):
            raise ValueError('Incomplete or duplicate top-k ranks')
        labels = [lookup.get((str(qid), str(r['record_id']))) for r in top]
        out = dict(data_version=version, query_id=qid, query_type=top[0]['query_type'],
                   judged_at_5=sum(x is not None for x in labels) / k)
        for scenario in ('unknown_as_nonmatch', 'unknown_as_match'):
            hits = [x == 2 or (x is None and scenario == 'unknown_as_match') for x in labels]
            first = next((i for i, hit in enumerate(hits, 1) if hit), None)
            out[scenario + '_precision_at_5'] = sum(hits) / k
            out[scenario + '_top1'] = float(hits[0])
            out[scenario + '_hit_at_5'] = float(any(hits))
            out[scenario + '_mrr_at_5'] = 1 / first if first else 0.
        result.append(out)
    summaries = []
    # Keep exact-name lookup separate: it often has only one correct record.
    for version in sorted({r['data_version'] for r in result}):
        for scope in ('item_intent', 'name_lookup'):
            subset = [r for r in result if r['data_version'] == version and
                      (r['query_type'] == 'missing_item_name') == (scope == 'name_lookup')]
            if subset:
                summaries.append(dict(data_version=version, scope=scope, queries=len(subset),
                    **{key: mean(r[key] for r in subset)
                       for key in subset[0] if key not in ('data_version', 'query_id', 'query_type')}))
    return result, summaries

per_query, summary = score_results(results.to_dict('records'), qrels.to_dict('records'))
summary_df = pd.DataFrame(summary)
display(summary_df)
pd.DataFrame(per_query).to_csv(root/'per_query_metrics_provisional.csv', index=False, encoding='utf-8-sig')
summary_df.to_csv(root/'summary_metrics_provisional.csv', index=False, encoding='utf-8-sig')
pool = results.drop_duplicates(['query_id','record_id'])[['query_id','query','query_type','record_id','merchant_name','original_items']].merge(qrels, on=['query_id','record_id'], how='left', validate='one_to_one').fillna('')
pool.to_csv(root/'pooled_results_for_review.csv', index=False, encoding='utf-8-sig')
qrels.to_csv(root/'prior_qrels.csv', index=False, encoding='utf-8-sig')
archive = root/'comparison_results_bundle.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['comparison_results.csv','run_manifest.json','per_query_metrics_provisional.csv',
                 'summary_metrics_provisional.csv','pooled_results_for_review.csv','prior_qrels.csv']:
        z.write(root/name, arcname=name)
print('미판정 후보:', pool.relevance.eq('').sum(), '/', len(pool))
print('현재 지표는 잠정 진단입니다. ZIP을 전달해 주세요.')
files.download(str(archive))


## 6. 자유 검색 — 선택
같은 검색어로 원본과 보강본의 상위 5개를 나란히 확인합니다. 단일 성공 사례로 전체 개선을 주장하지 않습니다.

In [ ]:
query = '국거리 고기'
q = model.encode([query], batch_size=1, max_length=MAX_LENGTH,
                 return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs'].astype(np.float32)[0]
q /= np.linalg.norm(q)
for version, vec in vectors.items():
    scores = vec @ q
    idx = np.argsort(-scores, kind='stable')[:5]
    result = data.iloc[idx][['record_id','merchant_name','original_items']].copy()
    result.insert(0,'score',scores[idx])
    print(version, query)
    display(result)


### 해석 및 다음 단계

1. 품목·자연어 의도 15개와 특정 상호명 5개를 분리해 봅니다.
2. 새 후보까지 동일한 qrels로 판정한 후 Precision@5, Top1, Hit@5, MRR@5 차이를 계산합니다.
3. 기존 질의에 맞춰 보강했으므로 이 결과는 개발 실험입니다. 팀 공통 신규 질의·공통 데이터에서 다시 검증해야 모델 선정 근거가 됩니다.
4. 전체 가맹점의 실제 판매 여부는 확인되지 않았으므로 Recall이나 확정 정답 기반 수치라고 부르지 않습니다.

모델 사용법 출처: https://huggingface.co/BAAI/bge-m3
